# Real DHI examples — extraction

Matches the F3 Demo 2023 project's own expert-picked real shallow bright spot location (OpendTect PickSet file, not generated by this project) onto the trace grid and extracts a patch in the same `.npz` format as the synthetic dataset (`synthetic_dhi_generation.ipynb`), so `dhi_detector.ipynb` (irp-nfs25) can evaluate the trained detector against a real anomaly, not only synthetic injections.

Source, from `F3_Demo_2023/Locations/`:
- `Shallow_Bright_Spot.pck` — a single real shallow bright spot's map-view polygon outline (31 vertices), reduced to its centroid as one representative point.

`il_extent=xl_extent=96` (not the synthetic set's 160) is a deliberate difference: at 160, zero real picks are patchable, since the real anomaly sits near the survey's inline edges. 96 was chosen because it's also Aziz's independently-proposed F4 patch-extent value — see `fixes-2026-07-29.md`.

(The chimney / non-chimney real-example set previously extracted here was dropped after evaluation showed it wasn't useful — see project notes.)

In [1]:
import sys
sys.path.append('..')

import segyio
import numpy as np
import pandas as pd

from src.dhi_pipeline import real_examples
from src.dhi_pipeline.horizons import build_coordinate_lookup
from src.dhi_pipeline.dataset import _read_subvolume

SEGY_PATH = '../data_raw/Seismic_data.sgy'
LOCATIONS_DIR = '../data_raw'
BS_PCK = f'{LOCATIONS_DIR}/Shallow_Bright_Spot.pck'

# same output location as the synthetic dataset (data/README.md, irp-nfs25) - patches/
# holds both synthetic (example_*.npz) and real (real_*.npz) examples, distinguished by filename
# NOTE: still a personal OneDrive path, not portable to another machine as-is - but the
# resulting real_brightspot_centroid.npz is already DVC-tracked in irp-nfs25's data.dvc,
# so the *output data* is reproducible via `dvc pull` even though this cell isn't.
OUTPUT_DIR = '/Users/nfs25/Library/CloudStorage/OneDrive-ImperialCollegeLondon/F3_synthetic_dhi_dataset'
IL_EXTENT = XL_EXTENT = 96

coords = build_coordinate_lookup(SEGY_PATH)
with segyio.open(SEGY_PATH, ignore_geometry=True) as f:
    inlines = sorted(set(f.attributes(segyio.TraceField.INLINE_3D)[:]))
    xlines = sorted(set(f.attributes(segyio.TraceField.CROSSLINE_3D)[:]))
    iline_map = {(int(il), int(xl)): i for i, (il, xl) in enumerate(zip(
        f.attributes(segyio.TraceField.INLINE_3D)[:], f.attributes(segyio.TraceField.CROSSLINE_3D)[:]))}
print(f"Coordinate lookup: {len(coords['ilxl_array'])} traces")

Coordinate lookup: 600515 traces


## Real shallow bright spot (single point, polygon centroid)

Not wrapped in a dedicated function — this is a single point reduced from a polygon outline, not a pickset file of individual picks, so it's built directly from the lower-level functions.

In [3]:
bs_picks = real_examples.load_pickset(BS_PCK)
print(f'{len(bs_picks)} polygon vertices loaded (expect 31)')

centroid = pd.DataFrame([{'x': bs_picks.x.mean(), 'y': bs_picks.y.mean(), 'time_ms': bs_picks.time_ms.mean()}])
matched = real_examples.match_picks_to_grid(centroid, coords['ilxl_array'], coords['xy_array'])
il_center, xl_center, center_time_ms = matched.loc[0, ['inline', 'crossline', 'time_ms']]
print(f'bright spot centroid -> il={il_center}, xl={xl_center}, time={center_time_ms:.1f}ms')

cache_inline_axis = np.arange(int(il_center - IL_EXTENT // 2), int(il_center + IL_EXTENT // 2))
cache_xl_axis = np.arange(int(xl_center - XL_EXTENT // 2), int(xl_center + XL_EXTENT // 2))

with segyio.open(SEGY_PATH, ignore_geometry=True) as f:
    samples_ms = f.samples.astype(float)
    dt_ms = float(samples_ms[1] - samples_ms[0])
    cached_subvol = _read_subvolume(f, iline_map, cache_inline_axis, cache_xl_axis, samples_ms.size)

patch = real_examples.extract_real_patch(
    il_center, xl_center, center_time_ms, cached_subvol,
    cache_inline_axis, cache_xl_axis, samples_ms, dt_ms, IL_EXTENT, XL_EXTENT,
)
real_patch_file = 'real_brightspot_centroid.npz'
np.savez_compressed(f'{OUTPUT_DIR}/patches/{real_patch_file}',
                     attribute_stack=patch['attribute_stack'], channel_names=np.array(patch['channel_names']))
print(f'saved {real_patch_file}')

31 polygon vertices loaded (expect 31)
bright spot centroid -> il=205.0, xl=1045.0, time=528.4ms
saved real_brightspot_centroid.npz
